<a href="https://colab.research.google.com/github/rameshjayamani-oss/GenAI_Assignment/blob/main/Assignment_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import pandas as pd
import numpy as np
from pathlib import Path

# -------------------------------------------------
# Original dataset
# -------------------------------------------------

original_data = {
    "Experience": [2, 5, 1, 8, 4, 10, 3, 6, 7, 2],
    "Training_Hours": [40, 60, 20, 80, 50, 90, 30, 70, 75, 25],
    "Working_Hours": [38, 42, 35, 45, 40, 48, 37, 44, 46, 36],
    "Projects": [3, 6, 2, 8, 5, 9, 4, 7, 7, 3],
    "Productivity_Score": [62, 78, 55, 88, 72, 92, 65, 82, 85, 60]
}

original_df = pd.DataFrame(original_data)

# -------------------------------------------------
# Set random seed for reproducible results
# -------------------------------------------------

np.random.seed(42)

# -------------------------------------------------
# Generate 40 synthetic records
# -------------------------------------------------

synthetic_records = []

for _ in range(40):

    # Generate realistic work-related features
    experience = np.random.randint(1, 11)

    training_hours = np.random.randint(
        max(20, experience * 8 - 5),
        min(101, experience * 9 + 16)
    )

    working_hours = np.random.randint(
        35,
        49
    )

    # Projects generally increase with experience
    projects = round(
        1.5 + (experience * 0.75) + np.random.normal(0, 0.8)
    )

    projects = int(np.clip(projects, 2, 10))

    # Generate productivity score using a realistic formula
    productivity_score = (
        45
        + (1.5 * experience)
        + (0.20 * training_hours)
        + (0.45 * projects)
        + (0.35 * working_hours)
        - (0.025 * max(0, working_hours - 44) ** 2)
        + np.random.normal(0, 2.5)
    )

    productivity_score = int(
        np.clip(round(productivity_score), 50, 98)
    )

    synthetic_records.append([
        experience,
        training_hours,
        working_hours,
        projects,
        productivity_score
    ])

synthetic_df = pd.DataFrame(
    synthetic_records,
    columns=[
        "Experience",
        "Training_Hours",
        "Working_Hours",
        "Projects",
        "Productivity_Score"
    ]
)

# -------------------------------------------------
# Combine original and synthetic data
# -------------------------------------------------

final_df = pd.concat(
    [original_df, synthetic_df],
    ignore_index=True
)
final_df

,Experience,Training_Hours,Working_Hours,Projects,Productivity_Score
0,2,40,38,3,62
1,5,60,42,6,78
2,1,20,35,2,55
3,8,80,45,8,88
4,4,50,40,5,72
5,10,90,48,9,92
6,3,30,37,4,65
7,6,70,44,7,82
8,7,75,46,7,85
9,2,25,36,3,60


In [4]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

# Define features (X) and target (y)
X = final_df[['Experience', 'Training_Hours', 'Working_Hours', 'Projects']]
y = final_df['Productivity_Score']

# Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("Data split into training and testing sets successfully.")

Data split into training and testing sets successfully.


In [5]:
# Initialize and train the Linear Regression model
model = LinearRegression()
model.fit(X_train, y_train)

print("Linear Regression model trained successfully.")

# Get the coefficients
coefficients_df = pd.DataFrame({'Coefficient': model.coef_}, index=X.columns)

# Assign to 'coefficients' as used in subsequent markdown cells
coefficients = coefficients_df

# Make predictions on the test set
y_pred = model.predict(X_test)

# Evaluate the model
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"\nModel Evaluation:")
print(f"Mean Squared Error (MSE): {mse:.2f}")
print(f"R-squared (R2): {r2:.2f}")

print("\nModel Coefficients:")
display(coefficients)

Linear Regression model trained successfully.

Model Evaluation:
Mean Squared Error (MSE): 7.19
R-squared (R2): 0.94

Model Coefficients:


,Coefficient
Experience,3.286508
Training_Hours,0.133949
Working_Hours,0.409698
Projects,-1.002384


In [6]:
corrMatrix = final_df.corr();
print(corrMatrix)

                    Experience  Training_Hours  Working_Hours  Projects  \
Experience            1.000000        0.954816       0.239094  0.967385   
Training_Hours        0.954816        1.000000       0.198286  0.929084   
Working_Hours         0.239094        0.198286       1.000000  0.249371   
Projects              0.967385        0.929084       0.249371  1.000000   
Productivity_Score    0.957901        0.937659       0.343010  0.925100   

                    Productivity_Score  
Experience                    0.957901  
Training_Hours                0.937659  
Working_Hours                 0.343010  
Projects                      0.925100  
Productivity_Score            1.000000  


1. Which factor most strongly impacts productivity?
To determine the factor with the strongest impact, we look at the correlation values as a proxy for the strength of linear relationships. Larger correlation values imply a stronger influence on productivity.

From the correlation matrix:

Experience (yrs): 0.958
Training Hours: 0.938
Working Hours: 0.343
Projects: 0.925
The factor with the highest correlation is Experience (yrs) (0.958), indicating it has the strongest positive impact on productivity among the features included. This suggests that for every additional year of experience, the productivity score is expected to increase the most relative to other factors.

2. How does training affect productivity?
The correlation for Training Hours (0.938) is also very strong and positive. This means that increasing training hours is associated with a significant increase in productivity, though slightly less than experience. Training provides a clear benefit to productivity, reinforcing the value of continued learning and skill development.

3. Should the company increase training hours or working hours?
Comparing the correlations:

Training Hours: 0.938
Working Hours: 0.343
Increasing Training Hours would likely provide a much greater boost to productivity than extending Working Hours, which appear to have a relatively weak positive relationship. While longer working hours might slightly increase productivity, the benefits of more training are considerably higher and likely more sustainable.

4. What happens if Working Hours increase beyond optimal limits?
The relatively low correlation (0.343) for working hours indicates a weak relationship with productivity. In reality, beyond an optimal point, increasing working hours may lead to fatigue and stress, reducing productivity—something a basic linear model or correlation cannot capture. This suggests there is an optimal range for working hours, after which productivity gains taper off or decline.

5. Can productivity ever decrease with more experience?
The strong positive correlation with experience (0.958) implies that, generally, productivity increases with more experience. However, real-world scenarios such as complacency, outdated skills, or decreased motivation might cause productivity to plateau or decline for very experienced employees. This nuance is not captured by a simple linear correlation.

6. How would you detect overfitting in this model?
Overfitting can be detected by:

Evaluating model performance on both training and separate test datasets. Significant performance drop on test data suggests overfitting.
Applying cross-validation techniques, such as K-fold, to verify the model’s ability to generalize.
Monitoring metrics like R-squared and mean squared error to ensure balanced performance.
7. Suggest one new feature to improve prediction accuracy.
A useful new feature could be Employee Satisfaction Score. Employee satisfaction is linked to motivation and engagement, factors that strongly influence productivity but are not captured by experience, training, working hours, or projects. Incorporating satisfaction can enhance the model's ability to predict productivity more accurately.
